# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rehman-dev288/FlyRank-AI-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# ML-04 — Search Intelligence Data Contract

## 1. Unit of Analysis + Time Window

* **Unit of Analysis (Grain):** One row represents one unique web page (`page_id`) aggregated over a 1-month observation window.
* **Time Window:** Mid-panel month `2026-03` (2026-03-01 to 2026-03-31) for feature engineering, evaluating target traffic shift in `2026-04`.
* **Data Source:** FlyRank Search Warehouse Parquet dataset (`hf://datasets/FlyRank/internship-warehouse/*.parquet`).

In [12]:
import duckdb
import os
import sys

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

hf_token = os.getenv("HF_TOKEN", "")
if hf_token:
    con.execute(f"SET s3_headers = '{{\"Authorization\": \"Bearer {hf_token}\"}}';")
else:
    print("ERROR: Hugging Face token (HF_TOKEN) not found. Please set it as a Colab secret for authenticated access to datasets.", file=sys.stderr)
    print("Refer to Colab's 'Secrets' panel (key icon on the left sidebar) to add HF_TOKEN.", file=sys.stderr)
    raise ValueError("HF_TOKEN is required for dataset access.")

query_window = """
SELECT
    MIN(date) as window_start,
    MAX(date) as window_end,
    COUNT(*) as total_raw_records
FROM 'hf://datasets/FlyRank/internship-warehouse/*.parquet'
WHERE strftime(date, '%Y-%m') = '2026-03'
"""
print(con.execute(query_window).df())

ERROR: Hugging Face token (HF_TOKEN) not found. Please set it as a Colab secret for authenticated access to datasets.
Refer to Colab's 'Secrets' panel (key icon on the left sidebar) to add HF_TOKEN.


ValueError: HF_TOKEN is required for dataset access.

## 2. Fields: Feature / Label / Context / Excluded

* **Feature Bucket:**
  * `avg_impressions`: Mean daily impressions in March 2026.
  * `avg_position`: Mean organic SERP ranking position in March 2026.
  * `click_volatility`: Coefficient of variation (STDDEV/AVG) of daily clicks.
  * `mid_month_ratio`: Mid-month traffic ratio (H2 impressions / H1 impressions).
  * `active_days`: Distinct days with >0 recorded clicks.
* **Label Bucket:** `traffic_decay_score` (Target proxy: relative percentage drop in average clicks during `2026-04` compared to `2026-03`).
* **Context Bucket:** `page_id`, `observation_month` (`2026-03`).
* **Excluded Bucket:** Raw daily query strings and pages with `< 10` total impressions.
  * **Why Excluded:** Removes search intent noise, private branded query exports, and high-variance long-tail unranked pages that distort model training.

In [ ]:
query_schema = """
SELECT column_name, data_type
FROM (DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/*.parquet')
LIMIT 10;
"""
print(con.execute(query_schema).df())

## 3. Verify It with Queries & Feature Frame

### 5-Feature Point-in-Time Justification:
1. `avg_impressions`: Knowable at decision moment because it aggregates only historical search impressions within `2026-03`.
2. `avg_position`: Knowable at decision moment because SERP position logs are recorded during the observation window prior to decision time.
3. `click_volatility`: Knowable at decision moment as it calculates past standard deviation over observed March click counts.
4. `mid_month_ratio`: Knowable at decision moment because it splits performance using internal cutoff dates within `2026-03`.
5. `active_days`: Knowable at decision moment because it counts historical days with positive click events prior to the end of March.

### Deliberate Leakage Trap:
An artificial column `leaked_future_clicks` (label proxy) was temporarily added to demonstrate artificial score inflation ($R^2 \approx 1.0$), then stripped to preserve honest model baseline numbers.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# 1. Grain Verification
q1_grain = """
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT page_id) as unique_pages,
    (COUNT(*) = COUNT(DISTINCT page_id)) as is_one_row_per_page
FROM (
    SELECT page_id FROM 'hf://datasets/FlyRank/internship-warehouse/*.parquet'
    WHERE strftime(date, '%Y-%m') = '2026-03'
    GROUP BY page_id
)
"""
print("--- Query 1: Grain Verification ---")
print(con.execute(q1_grain).df())

# 2. Slice & Date Span
q2_slice = """
SELECT
    MIN(date) as min_date,
    MAX(date) as max_date,
    COUNT(*) as total_records
FROM 'hf://datasets/FlyRank/internship-warehouse/*.parquet'
WHERE strftime(date, '%Y-%m') = '2026-03'
"""
print("\n--- Query 2: Slice & Date Span ---")
print(con.execute(q2_slice).df())

# 3. Availability Check (IS TRUE)
q3_availability = """
SELECT
    COUNT(*) as total_rows,
    COUNT(CASE WHEN (impressions >= 10 AND clicks IS NOT NULL) IS TRUE THEN 1 END) as valid_rows
FROM 'hf://datasets/FlyRank/internship-warehouse/*.parquet'
WHERE strftime(date, '%Y-%m') = '2026-03'
"""
print("\n--- Query 3: Availability (IS TRUE) ---")
print(con.execute(q3_availability).df())

# 4. Feature Extraction & Leakage Experiment
q_features = """
SELECT
    page_id,
    AVG(impressions) as avg_impressions,
    AVG(position) as avg_position,
    (STDDEV(clicks) / NULLIF(AVG(clicks), 0)) as click_volatility,
    (AVG(CASE WHEN date >= '2026-03-15' THEN impressions END) /
     NULLIF(AVG(CASE WHEN date < '2026-03-15' THEN impressions END), 0)) as mid_month_ratio,
    COUNT(DISTINCT CASE WHEN clicks > 0 THEN date END) as active_days,
    AVG(clicks) as target_label
FROM 'hf://datasets/FlyRank/internship-warehouse/*.parquet'
WHERE strftime(date, '%Y-%m') = '2026-03'
GROUP BY page_id
HAVING AVG(impressions) >= 10
"""
df = con.execute(q_features).df().fillna(0)

# Leakage Trap
df['leaked_future_clicks'] = df['target_label'] * 1.05
X_leaked = df[['avg_impressions', 'avg_position', 'click_volatility', 'mid_month_ratio', 'active_days', 'leaked_future_clicks']]
y = df['target_label']

model = RandomForestRegressor(random_state=42)
model.fit(X_leaked, y)
print(f"\n[LEAKAGE TRAP] Model R2 Score with Leaked Column: {r2_score(y, model.predict(X_leaked)):.4f}")

# Honest Baseline (X_honest Defined Here)
X_honest = df[['avg_impressions', 'avg_position', 'click_volatility', 'mid_month_ratio', 'active_days']]
model.fit(X_honest, y)
print(f"[HONEST BASELINE] Model R2 Score without Leakage: {r2_score(y, model.predict(X_honest)):.4f}")

## 4. Data Limits

* **Named Limitation:** This slice relies exclusively on historical Search Console aggregations within `2026-03`.
* **Unobserved External Drivers:** It cannot observe search engine core updates, technical downtime, backlink profile changes, SERP UI layout updates, or seasonal demand shifts outside the cutoff window.
* **Observed Scope:** Model predictions represent decision-support signals based strictly on past observed search performance.

In [ ]:
print("Feature missing value audit:")
print(X_honest.isnull().sum())

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.